# Importing the dataset for inspection.

In [14]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('nyu-mll/glue', 'mrpc')
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

# Looking at the sentences in the dataset for quick analysis.

In [15]:
train_sample_set = dataset["train"][:20]

for i in range(len(train_sample_set["sentence1"])):
    print(f"Sample {i+1}:")
    print(f"Sentence 1: {train_sample_set['sentence1'][i]}")
    print(f"Sentence 2: {train_sample_set['sentence2'][i]}")
    print(f"Label: {train_sample_set['label'][i]}")
    print(f"ID: {train_sample_set['idx'][i]}")
    print()

Sample 1:
Sentence 1: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
Sentence 2: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
Label: 1
ID: 0

Sample 2:
Sentence 1: Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
Sentence 2: Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .
Label: 0
ID: 1

Sample 3:
Sentence 1: They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
Sentence 2: On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .
Label: 1
ID: 2

Sample 4:
Sentence 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Sentence 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing 

# Creating split sets to check for overlap of duplicates across the splits.

In [16]:
full_train_sample_set = dataset["train"][:]

full_test_sample_set = dataset["test"][:]

full_valid_sample_set = dataset["validation"][:]

In [17]:
train_sentences = full_train_sample_set["sentence1"] + full_train_sample_set["sentence2"]

test_sentences = full_test_sample_set["sentence1"] + full_test_sample_set["sentence2"]

valid_sentences = full_valid_sample_set["sentence1"] + full_valid_sample_set["sentence2"]

train_set = set(train_sentences)
valid_set = set(valid_sentences)
test_set = set(test_sentences)

train_valid_overlap = train_set.intersection(valid_set)
train_test_overlap = train_set.intersection(test_set)
valid_test_overlap = valid_set.intersection(test_set)

print(len(train_valid_overlap))
print(len(train_test_overlap))
print(len(valid_test_overlap))

43
244
24


In [18]:
for sentence in sorted(train_test_overlap)[:10]:
    print(sentence)

" Arifin has disclosed . . . that he is involved with a group of like-minded individuals in planning terrorist attacks against certain targets in Thailand .
" But they did not , as of the time of this particular tragic event , provide the security we had requested , " Mr Jordan said .
" For us , that doesn 't make a difference - the sexual orientation , " Archbishop Tutu said in the black urban centre of Soweto .
" Further testing is still under way , but at this stage , given the early detection , the outlook in such instances would be positive , " the specialist said yesterday .
" However , we are not certain about the outcome of the discussion at this moment . "
" I ´ m very proud of the citizens of this state , " Gov. John Baldacci said after votes from Tuesday ´ s referendum were counted .
" In fact , I was physically sick several times at this stage , because he looked so desperate , " she said .
" It remains to be seen whether the revenue recovery will be short- or long-lived , 

### Data issues found during manual inspection

1. **Pre-tokenization spacing artifacts:** Punctuation, quotation marks, possessives, and currency symbols are separated by spaces, for example `" the witness "`, `Dominick 's`, and `$ 2.5 billion`.

2. **Numerical/factual disagreement inside a positive pair:** Sample 5 is labeled paraphrase even though one sentence says the stock rose `$ 2.11`, or `11 percent`, to `$ 21.51`, while the other says `$ 1.63`, or `8 percent`, to `$ 21.03`. This shows that some positive pairs preserve the same general event while differing on specific facts.

3. **Entity/reference differences:** In Sample 3, one sentence uses `"They"` and `"the cargo"` while the other uses `"the ship 's owners"` and `"the explosives"`, but the pair is still labeled paraphrase. These reference changes can make semantic equivalence harder to judge.

4. **Potential label ambiguity:** Sample 17 is labeled non-paraphrase even though both sentences express nearly the same core event about chaos allowing damaging logging and fishing operations. The extra `"dolphin scheme"` clause may explain the negative label, but the pair is borderline.

5. **Repeated sentences across published splits:** The same sentence can appear in more than one split. This creates sentence-level leakage, which could make model performance appear more optimistic because some test sentences were already seen during training.

# Label Counts

In [19]:
train_label_0 = (pd.Series(full_train_sample_set["label"]) == 0).sum()
train_label_1 = (pd.Series(full_train_sample_set["label"]) == 1).sum()
train_label_total = train_label_0 + train_label_1
train_majority_pct = max(train_label_0, train_label_1)/train_label_total
train_majority_class = 0 if train_label_0 > train_label_1 else 1

test_label_0 = (pd.Series(full_test_sample_set["label"]) == 0).sum()
test_label_1 = (pd.Series(full_test_sample_set["label"]) == 1).sum()
test_label_total = test_label_0 + test_label_1
test_majority_pct = max(test_label_0, test_label_1)/test_label_total
test_majority_class = 0 if test_label_0 > test_label_1 else 1

valid_label_0 = (pd.Series(full_valid_sample_set["label"]) == 0).sum()
valid_label_1 = (pd.Series(full_valid_sample_set["label"]) == 1).sum()
valid_label_total = valid_label_0 + valid_label_1   
valid_majority_pct = max(valid_label_0, valid_label_1)/valid_label_total
valid_majority_class = 0 if valid_label_0 > valid_label_1 else 1


In [20]:
class_summary_df = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'label_0': [train_label_0, valid_label_0, test_label_0],
    'label_1': [train_label_1, valid_label_1, test_label_1],
    'total': [train_label_total, valid_label_total, test_label_total],
    'majority_class': [train_majority_class, valid_majority_class, test_majority_class],
    'majority_pct': [train_majority_pct, valid_majority_pct, test_majority_pct]
})

class_summary_df

,split,label_0,label_1,total,majority_class,majority_pct
0,train,1194,2474,3668,1,0.674482
1,validation,129,279,408,1,0.683824
2,test,578,1147,1725,1,0.664928


Across all three splits, label 1 (paraphrase) is the majority class. It represents about 67.45% of the training set, 68.38% of the validation set, and 66.49% of the test set. Because the class distribution is moderately imbalanced toward paraphrases, accuracy alone could be misleading, so F1 is also important for evaluation.